# MC_BOOTSTRAP_001 — three-class proposal model
Enable a Kaggle GPU and attach the private MC bootstrap bundle. Training is configured for 150 epochs and saves `epoch0.pt` through `epoch149.pt`. After every completed epoch, the runner atomically refreshes `/kaggle/working/MC_BOOTSTRAP_001_LATEST_RESUME.zip`. This model generates review/demo proposals; it is not human-approved MC_001 final ground truth.


## Optional resume
For a new Kaggle session, attach a private dataset containing the previously downloaded `MC_BOOTSTRAP_001_LATEST_RESUME.zip`. The runner verifies experiment identity, dataset/config hashes, checkpoint hashes, and completed epoch, then automatically continues from the newest valid archive. Do not rename or edit files inside the archive.


In [ ]:
from pathlib import Path
import zipfile
input_root=Path('/kaggle/input')
requirements=list(input_root.rglob('requirements-kaggle.txt'))
if requirements:
    assert len(requirements)==1, f'STOP: expected one extracted bundle, found {len(requirements)}'
    bundle_root=requirements[0].parent
else:
    bundle_archives=list(input_root.rglob('mc_bootstrap_001_kaggle_bundle.zip'))
    assert len(bundle_archives)==1, f'STOP: expected one bundle ZIP, found {len(bundle_archives)}'
    bundle_root=Path('/kaggle/working/mc_bootstrap_bundle')
    bundle_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle_archives[0]) as archive:
        assert archive.testzip() is None, 'STOP: corrupt bundle ZIP'
        archive.extractall(bundle_root)
requirements=[bundle_root/'requirements-kaggle.txt']
print('Bundle root:', bundle_root)
!pip install -q -r {requirements[0]}


In [ ]:
resume_sources=[p for p in Path('/kaggle/input').rglob('*.zip') if p.name in {'MC_BOOTSTRAP_001_LATEST_RESUME.zip','MC_BOOTSTRAP_001_RESUME.zip'}]
resume_sources += list(Path('/kaggle/input').rglob('resume_metadata.json'))
print('Resume source:', resume_sources if resume_sources else 'none — starting epoch 1')


In [ ]:
import runpy, sys
runner=bundle_root/'mc_bootstrap_kaggle_runner.py'
assert runner.is_file(), f'STOP: missing runner at {runner}'
sys.argv=[str(runner), '--bundle', str(bundle_root)]
runpy.run_path(str(runner), run_name='__main__')


Download `/kaggle/working/MC_BOOTSTRAP_001_RECOVERY.zip` after completion. During a long run, periodically download `/kaggle/working/MC_BOOTSTRAP_001_LATEST_RESUME.zip`; a hard Kaggle session loss cannot preserve files that were never downloaded or committed as notebook output. Historical per-epoch checkpoints remain under `/kaggle/working/MC_BOOTSTRAP_001/weights/`. Preserve `run_metadata.json`; all weights remain proposal-only until human review and final MC_001 training.
